In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
dataset_path = "/kaggle/input/datasets/sabahesaraki/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT"
classes = ["benign","malignant","normal"]

In [5]:
image_paths = []
labels = []
for c in classes:
    folder = os.path.join(dataset_path,c)
    for file in os.listdir(folder):
        if "mask" not in file:
            image_paths.append(os.path.join(folder,file))
            labels.append(c)
print("Total images:",len(image_paths))

Total images: 780


In [6]:
df = pd.DataFrame({
    "filename":image_paths,
    "class":labels
})
df["class"].value_counts()

class
benign       437
malignant    210
normal       133
Name: count, dtype: int64

In [7]:
train_df, temp_df = train_test_split(df,test_size=0.30,stratify=df["class"],random_state=42)
val_df, test_df = train_test_split(temp_df,test_size=0.50,stratify=temp_df["class"],random_state=42)

In [8]:
IMG_SIZE = 224
BATCH_SIZE = 16
train_gen = ImageDataGenerator(rescale=1./255,rotation_range=15,zoom_range=0.1,width_shift_range=0.1,height_shift_range=0.1,horizontal_flip=True)
test_gen = ImageDataGenerator(rescale=1./255)

In [9]:
train_data = train_gen.flow_from_dataframe(train_df,x_col="filename",y_col="class",target_size=(IMG_SIZE,IMG_SIZE),
                batch_size=BATCH_SIZE,class_mode="categorical")
val_data = test_gen.flow_from_dataframe(val_df,x_col="filename",y_col="class",target_size=(IMG_SIZE,IMG_SIZE),
                batch_size=BATCH_SIZE,class_mode="categorical")
test_data = test_gen.flow_from_dataframe(test_df,x_col="filename",y_col="class",target_size=(IMG_SIZE,IMG_SIZE),
                batch_size=BATCH_SIZE,class_mode="categorical",shuffle=False)

Found 546 validated image filenames belonging to 3 classes.
Found 117 validated image filenames belonging to 3 classes.
Found 117 validated image filenames belonging to 3 classes.


In [10]:
base_model = VGG16(weights="imagenet",include_top=False,input_shape=(224,224,3))
for layer in base_model.layers:
    layer.trainable = False
x = Flatten()(base_model.output)
x = Dense(256,activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(3,activation="softmax")(x)
model = Model(inputs=base_model.input,outputs=output)
model.compile(optimizer=Adam(learning_rate=0.0001),loss="categorical_crossentropy", metrics=["Accuracy"])

I0000 00:00:1773553091.689089     103 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [11]:
history = model.fit(train_data,validation_data=val_data,epochs=15)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15


I0000 00:00:1773553101.909067     162 service.cc:152] XLA service 0x798d5c00d0d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773553101.909105     162 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773553102.378343     162 cuda_dnn.cc:529] Loaded cuDNN version 91002


 1/35 ━━━━━━━━━━━━━━━━━━━━ 4:21 8s/step - Accuracy: 0.3750 - loss: 0.9741

I0000 00:00:1773553107.834232     162 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


35/35 ━━━━━━━━━━━━━━━━━━━━ 28s 601ms/step - Accuracy: 0.4509 - loss: 1.2081 - val_Accuracy: 0.5897 - val_loss: 0.8353
Epoch 2/15
35/35 ━━━━━━━━━━━━━━━━━━━━ 10s 297ms/step - Accuracy: 0.6434 - loss: 0.8439 - val_Accuracy: 0.6923 - val_loss: 0.7341
Epoch 3/15
35/35 ━━━━━━━━━━━━━━━━━━━━ 10s 297ms/step - Accuracy: 0.6070 - loss: 0.8317 - val_Accuracy: 0.7521 - val_loss: 0.6824
Epoch 4/15
35/35 ━━━━━━━━━━━━━━━━━━━━ 10s 295ms/step - Accuracy: 0.6459 - loss: 0.7295 - val_Accuracy: 0.7265 - val_loss: 0.6341
Epoch 5/15
35/35 ━━━━━━━━━━━━━━━━━━━━ 10s 300ms/step - Accuracy: 0.6256 - loss: 0.8008 - val_Accuracy: 0.7949 - val_loss: 0.6112
Epoch 6/15
35/35 ━━━━━━━━━━━━━━━━━━━━ 10s 290ms/step - Accuracy: 0.6623 - loss: 0.7118 - val_Accuracy: 0.7179 - val_loss: 0.6707
Epoch 7/15
35/35 ━━━━━━━━━━━━━━━━━━━━ 11s 303ms/step - Accuracy: 0.6790 - loss: 0.7139 - val_Accuracy: 0.7778 - val_loss: 0.5727
Epoch 8/15
35/35 ━━━━━━━━━━━━━━━━━━━━ 11s 307ms/step - Accuracy: 0.7369 - loss: 0.6137 - val_Accuracy: 0.777

In [12]:
pred = model.predict(test_data)
pred_classes = np.argmax(pred,axis=1)
true_classes = test_data.classes
print(classification_report(true_classes,pred_classes))

8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 314ms/step
              precision    recall  f1-score   support

           0       0.75      0.73      0.74        66
           1       0.65      0.65      0.65        31
           2       0.64      0.70      0.67        20

    accuracy                           0.70       117
   macro avg       0.68      0.69      0.68       117
weighted avg       0.70      0.70      0.70       117



Now applying Random over sampling

In [13]:
print(train_df["class"].value_counts())

class
benign       306
malignant    147
normal        93
Name: count, dtype: int64


In [14]:
max_count = train_df["class"].value_counts().max()
oversampled_data = []
for c in train_df["class"].unique():
    subset = train_df[train_df["class"] == c]
    subset_over = subset.sample(max_count,replace=True,random_state=42)
    oversampled_data.append(subset_over)
train_df_over = pd.concat(oversampled_data)
train_df_over = train_df_over.sample(frac=1).reset_index(drop=True)

In [15]:
print(train_df_over["class"].value_counts())

class
benign       306
malignant    306
normal       306
Name: count, dtype: int64


Now the dataset is balanced

In [16]:
train_data_over = train_gen.flow_from_dataframe(train_df_over,x_col="filename",y_col="class",
                  target_size=(224,224), batch_size=16,class_mode="categorical")

Found 918 validated image filenames belonging to 3 classes.


In [18]:
base_model = VGG16(weights="imagenet",include_top=False,input_shape=(224,224,3))
for layer in base_model.layers:
    layer.trainable = False
x = Flatten()(base_model.output)
x = Dense(256,activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(3,activation="softmax")(x)
model = Model(inputs=base_model.input,outputs=output)
model.compile(optimizer=Adam(learning_rate=0.0001),loss="categorical_crossentropy", metrics=["Accuracy"])

In [19]:
history = model.fit(train_data_over,validation_data=val_data,epochs=15)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 23s 358ms/step - Accuracy: 0.4017 - loss: 1.1797 - val_Accuracy: 0.7009 - val_loss: 0.7904
Epoch 2/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 17s 290ms/step - Accuracy: 0.5826 - loss: 0.8659 - val_Accuracy: 0.5897 - val_loss: 0.7860
Epoch 3/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 17s 291ms/step - Accuracy: 0.6696 - loss: 0.7431 - val_Accuracy: 0.6496 - val_loss: 0.7290
Epoch 4/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 17s 286ms/step - Accuracy: 0.6982 - loss: 0.6998 - val_Accuracy: 0.6239 - val_loss: 0.7151
Epoch 5/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 17s 293ms/step - Accuracy: 0.7624 - loss: 0.5745 - val_Accuracy: 0.6667 - val_loss: 0.6906
Epoch 6/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 17s 286ms/step - Accuracy: 0.7864 - loss: 0.5421 - val_Accuracy: 0.7436 - val_loss: 0.6166
Epoch 7/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 16s 284ms/step - Accuracy: 0.7791 - loss: 0.5464 - val_Accuracy: 0.7436 - val_loss: 0.6367
Epoch 8/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 16s 283ms/step - Accuracy: 0.7631 - loss: 0.5320 - val_Accu

In [20]:
pred = model.predict(test_data)
pred_classes = np.argmax(pred, axis=1)
true_classes = test_data.classes
print(classification_report(true_classes, pred_classes))

8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 191ms/step
              precision    recall  f1-score   support

           0       0.77      0.67      0.72        66
           1       0.60      0.68      0.64        31
           2       0.64      0.80      0.71        20

    accuracy                           0.69       117
   macro avg       0.67      0.71      0.69       117
weighted avg       0.70      0.69      0.69       117

